<a href="https://colab.research.google.com/github/niofrequency/google-collab_image/blob/main/IMG2VID-WAN22-A14B(faster).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Image to Video — Wan 2.2 A14B (best quality, L4 / A100)

Turns a still image into a short video with **Wan 2.2 Image-to-Video A14B**, the best open image-to-video model
for Colab. Needs an **L4 or A100** GPU (compute units). On the free T4 use
[IMG2VID-WAN22-5B-FREE.ipynb](https://colab.research.google.com/github/niofrequency/google-collab_image/blob/main/IMG2VID-WAN22-5B-FREE.ipynb).

1. **Runtime → Change runtime type → L4 GPU** (or A100).
2. Run **cell 1** (install + downloads, ~25 GB on an L4 — several minutes), **cell 2** (saves the server program) and
   **cell 3** (starts it and prints the `READY:` URL).
3. On the website open the **Image to Video** tab and paste the URL.

*Fast* mode uses the 4-step Lightning LoRAs (a few minutes per 5-second clip on an L4); *Quality* mode uses 20 steps
(much slower). Cell 1 picks Q4_K_M on an L4 and Q8_0 on an A100. Cell 4 shows the server log and memory use.
Don't run another GPU notebook at the same time.

**Sound:** cell 1 also installs **MMAudio**, which can add matching sound effects and ambience (footsteps, waves, wind, engines…) to each video — turn on *Add sound* on the website. It takes about another minute per clip. MMAudio is for non-commercial use (CC BY-NC 4.0); set `ADD_SOUND = False` in cell 1 to skip it.

In [1]:
# Cell 1: install ComfyUI, SageAttention, check the GPU and download Native FP8 Wan 2.2 I2V A14B
import os, json, subprocess

os.makedirs("/content", exist_ok=True)
%cd /content

FAMILY = "a14b"
!git clone -q https://github.com /content/ComfyUI || true
!cd /content/ComfyUI && git pull -q || true
!cd /content/ComfyUI && pip install -q -r requirements.txt
!pip install -q fastapi uvicorn requests psutil sageattention
!apt-get update -qq && apt-get install -qqy aria2 ffmpeg > /dev/null

# ---- Google Drive LoRA sync (disabled for Vast.ai)
USE_GOOGLE_DRIVE = False
USER_LORAS = "/content/ComfyUI/models/loras/user"
os.makedirs("/content/ComfyUI/models/loras", exist_ok=True)
os.makedirs(USER_LORAS, exist_ok=True)

# ---- GPU Check (fixed list vs string bug)
try:
    gpu_lines = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
        capture_output=True, text=True
    ).stdout.strip().split("\n")
    gpu = gpu_lines[0] if gpu_lines else ""
except FileNotFoundError:
    gpu = ""

if "," not in gpu:
    raise RuntimeError("This runtime has no GPU detected via nvidia-smi.")
gpu_name, vram_mib = [x.strip() for x in gpu.rsplit(",", 1)]
GPU = {"name": gpu_name, "vram_gib": round(int(vram_mib) / 1024, 1)}
if GPU["vram_gib"] < 20:
    raise RuntimeError(f"GPU ({GPU['name']}, {GPU['vram_gib']} GB) is too small for A14B. Use L4 or A100.")

# ---- Native ComfyUI FP8 Scaled Models (2x faster than GGUF, sharper 480p detail)
HF_WAN = "https://huggingface.co/Comfy-Org/Wan-2.2/resolve/main/split_files"
FILES = {
    "high": ("wan2.2_i2v_high_noise_14B_fp8_scaled.safetensors", f"{HF_WAN}/diffusion_models/wan2.2_i2v_high_noise_14B_fp8_scaled.safetensors", "diffusion_models"),
    "low": ("wan2.2_i2v_low_noise_14B_fp8_scaled.safetensors", f"{HF_WAN}/diffusion_models/wan2.2_i2v_low_noise_14B_fp8_scaled.safetensors", "diffusion_models"),
    "lora_high": ("wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors", f"{HF_WAN}/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors", "loras"),
    "lora_low": ("wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors", f"{HF_WAN}/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors", "loras"),
    "text_encoder": ("umt5_xxl_fp8_e4m3fn_scaled.safetensors", f"{HF_WAN}/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors", "text_encoders"),
    "vae": ("wan_2.1_vae.safetensors", f"{HF_WAN}/vae/wan_2.1_vae.safetensors", "vae"),
}
DEFAULTS = {
    "fps": 16, "max_frames": 121, "seconds": 5, "size": "480p",
    "fast": {"steps": 4, "cfg": 1.0, "shift": 5.0, "sampler": "euler", "scheduler": "simple"},
    # Quality: 12 steps with uni_pc/beta on native FP8 gives crisp 480p clarity quickly
    "quality": {"steps": 12, "cfg": 4.0, "shift": 7.0, "sampler": "uni_pc", "scheduler": "beta"},
    # Balanced: 2 CFG steps for anatomy + 6 Lightning steps (best WaveSpeed-like speed/quality mix)
    "balanced": {"steps": 8, "cfg": 4.0, "shift": 5.0, "sampler": "euler", "scheduler": "simple"},
    "default_mode": "balanced",
}
LABEL = "Wan 2.2 I2V A14B (Native FP8 + SageAttention)"

# ---- Sound (optional): MMAudio adds matching sound effects / ambience to finished videos.
#      Set ADD_SOUND = False to skip it (saves ~8 GB of downloads). Model licence: CC BY-NC 4.0 (non-commercial).
ADD_SOUND = True
SOUND_OK = False
if ADD_SOUND:
    print("Installing MMAudio (sound for videos)...")
    !test -d /content/MMAudio || git clone -q https://github.com/hkchengrex/MMAudio /content/MMAudio
    !cd /content/MMAudio && git fetch -q origin && git checkout -q 974010a
    !pip install -q torchdiffeq open_clip_torch timm av einops librosa omegaconf colorlog
    try:
        from huggingface_hub import snapshot_download
        snapshot_download("apple/DFN5B-CLIP-ViT-H-14-384", allow_patterns=["open_clip*"])
        snapshot_download("nvidia/bigvgan_v2_44khz_128band_512x", allow_patterns=["config.json", "bigvgan_generator.pt"])
        subprocess.run(["python", "-c", "from mmaudio.eval_utils import all_model_cfg as c; c['large_44k_v2'].download_if_needed()"],
                       cwd="/content/MMAudio", check=True)
        SOUND_OK = True
    except Exception as e:
        print(f"Sound is turned off: MMAudio could not be set up ({e}). Videos still work, just silent.")

# ---- Download (resumes if interrupted) and write the config the server reads
for key, (name, url, folder) in FILES.items():
    print(f"Downloading {name}...")
    os.makedirs(f"/content/ComfyUI/models/{folder}", exist_ok=True)
    subprocess.run(["aria2c", "--console-log-level=error", "-c", "-x", "16", "-s", "16", "-k", "1M", url,
                    "-d", f"/content/ComfyUI/models/{folder}", "-o", name], check=True)

json.dump({"family": FAMILY, "label": LABEL, "gpu": GPU, "defaults": DEFAULTS, "sound": SOUND_OK,
           "files": {k: v[0] for k, v in FILES.items()}}, open("/content/video_config.json", "w"), indent=1)
print(f"GPU: {GPU['name']} ({GPU['vram_gib']} GB) -> {LABEL}, sound {'on' if SOUND_OK else 'off'}. Now run cells 2 and 3.")


/content
fatal: repository 'https://github.com/' not found
/bin/bash: line 1: cd: /content/ComfyUI: No such file or directory
/bin/bash: line 1: cd: /content/ComfyUI: No such file or directory


Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive


RuntimeError: This Colab runtime has no GPU. Switch to L4 or A100.

In [ ]:
logging.getLogger("root").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

# Enable SageAttention & fast TF32 matrix math BEFORE importing nodes
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
import comfy.cli_args
comfy.cli_args.args.use_sage_attention = True

import nodes
import comfy.utils
import comfy.model_management
from comfy_extras import nodes_wan, nodes_model_advanced

M = nodes.NODE_CLASS_MAPPINGS
UNETLoader, CLIPLoader = M["UNETLoader"](), M["CLIPLoader"]()
VAELoader, LoraLoaderModelOnly = M["VAELoader"](), M["LoraLoaderModelOnly"]()
CLIPTextEncode, VAEDecode = M["CLIPTextEncode"](), M["VAEDecode"]()
KSampler, KSamplerAdvanced = M["KSampler"](), M["KSamplerAdvanced"]()
ModelSamplingSD3 = nodes_model_advanced.ModelSamplingSD3()
F = CONFIG["files"]
D = CONFIG["defaults"]

# Load Native FP8 models (much faster than GGUF on L4/A100)
print(f"Loading {CONFIG['label']}...", flush=True)
with torch.inference_mode():
    clip = CLIPLoader.load_clip(F["text_encoder"], "wan")[0]
    vae = VAELoader.load_vae(F["vae"])[0]
    if FAMILY == "a14b":
        # Use fp8_e4m3fn_fast on L4 (Ada Lovelace) for hardware FP8 tensor core acceleration
        weight_dtype = "fp8_e4m3fn_fast" if torch.cuda.get_device_capability() == (8, 9) else "default"
        base_high = UNETLoader.load_unet(F["high"], weight_dtype)[0]
        base_low = UNETLoader.load_unet(F["low"], weight_dtype)[0]
        fast_high = LoraLoaderModelOnly.load_lora_model_only(base_high, F["lora_high"], 1.0)[0]
        fast_low = LoraLoaderModelOnly.load_lora_model_only(base_low, F["lora_low"], 1.0)[0]
        MODELS = {
            "fast": tuple(ModelSamplingSD3.patch(m, D["fast"]["shift"])[0] for m in (fast_high, fast_low)),
            "quality": tuple(ModelSamplingSD3.patch(m, D["quality"]["shift"])[0] for m in (base_high, base_low)),
            "balanced": tuple(ModelSamplingSD3.patch(m, D["balanced"]["shift"])[0] for m in (base_high, fast_high, fast_low)),
        }
    else:
        base = UNETLoader.load_unet(F["model"], "default")[0]
        fast = LoraLoaderModelOnly.load_lora_model_only(base, F["lora_turbo"], 1.0)[0]
        MODELS = {
            "fast": ModelSamplingSD3.patch(fast, D["fast"]["shift"])[0],
            "quality": ModelSamplingSD3.patch(base, D["quality"]["shift"])[0],
        }
print("Models loaded.", flush=True)


In [ ]:
# Cell 3: start the generation server as a separate program, plus the Cloudflare tunnel.
# If the server crashes (e.g. runs out of memory) only that program dies: it's restarted automatically
# with the same URL, and this notebook keeps running. Re-running this cell restarts the server with the
# current code from cell 2. To stop everything: Runtime → Restart session.
import os, re, sys, time, subprocess, threading, urllib.request

SERVER_SCRIPT = "/content/video_server.py"
SERVER_LOG = "/content/video_server.log"
CRASH_FILE = "/content/server_crash.txt"
PORT = 8000
CLOUDFLARED = "/usr/local/bin/cloudflared"


def server_up():
    try:
        return urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=3).status == 200
    except Exception:
        return False


def log_tail(n=15):
    try:
        return "".join(open(SERVER_LOG, errors="replace").readlines()[-n:])
    except OSError:
        return ""


class Supervisor:
    """Runs the server program and restarts it when it crashes (gives up after 4 crashes in 10 minutes)."""

    def __init__(self):
        self.proc, self.thread, self.stopping, self.gave_up, self.crashes = None, None, False, False, []

    def _loop(self):
        while not self.stopping:
            with open(SERVER_LOG, "a") as log:
                log.write(f"\n===== {time.strftime('%H:%M:%S')} starting the server =====\n")
                log.flush()
                self.proc = subprocess.Popen([sys.executable, "-u", SERVER_SCRIPT], cwd="/content/ComfyUI",
                                             stdout=log, stderr=subprocess.STDOUT,
                                             env={**os.environ, "PYTHONUNBUFFERED": "1"},
                                             start_new_session=True)  # stopping a notebook cell must not stop the server
                code = self.proc.wait()
            if self.stopping:
                break
            why = ("ran out of memory and was stopped by the system" if code in (-9, 137)
                   else f"crashed (exit code {code})")
            self.crashes = [t for t in self.crashes if time.time() - t < 600] + [time.time()]
            if len(self.crashes) >= 4:
                msg = (f"The generation server {why} 4 times in 10 minutes, so it was not restarted. "
                       f"Check the log (cell 4), then run cell 3 again.")
                self.gave_up = True
            else:
                msg = (f"The generation server {why} at {time.strftime('%H:%M')} and was restarted "
                       f"(models reload in 1-3 minutes). Try a smaller size, fewer pictures or batch 1.")
            with open(CRASH_FILE, "w") as fh:
                fh.write(msg)
            with open(SERVER_LOG, "a") as log:
                log.write(f"\n!!!!! {msg}\n")
            if self.gave_up:
                break
            time.sleep(3)

    def start(self):
        self.stopping, self.gave_up = False, False
        self.thread = threading.Thread(target=self._loop, daemon=True)
        self.thread.start()

    def stop(self):
        self.stopping = True
        if self.proc and self.proc.poll() is None:
            self.proc.terminate()
            try:
                self.proc.wait(30)
            except subprocess.TimeoutExpired:
                self.proc.kill()
        if self.thread:
            self.thread.join(35)


# (Re)start the server program
if "SUPERVISOR" in globals():
    print("Restarting the server with the current code...")
    try:
        SUPERVISOR.stop()
    except Exception:
        pass
if os.path.exists(CRASH_FILE):
    os.remove(CRASH_FILE)
SUPERVISOR = Supervisor()
SUPERVISOR.start()

print("Loading the video models in the server program (a few minutes)...")
t0 = time.time()
while not server_up():
    if SUPERVISOR.gave_up or time.time() - t0 > 900:
        print("The server did not start. Last lines of its log:\n" + log_tail(30))
        raise RuntimeError("Server failed to start — see the log above")
    time.sleep(5)
print(f"Server is up after {time.time() - t0:.0f}s.")

# Cloudflare tunnel (started once; the URL stays the same when the server restarts).
# If it stopped (e.g. the runtime hiccuped), a new one is started — with a NEW URL to paste into the website.
if globals().get("TUNNEL_URL") and (globals().get("TUNNEL_PROC") is None or TUNNEL_PROC.poll() is not None):
    print("The old tunnel stopped, so the old URL no longer works — starting a new one (new URL below).")
    TUNNEL_URL = None
if not globals().get("TUNNEL_URL"):
    subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
    if not os.path.exists(CLOUDFLARED):
        subprocess.run(["wget", "-q", "-O", CLOUDFLARED,
                        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
        os.chmod(CLOUDFLARED, 0o755)
    TUNNEL_PROC = subprocess.Popen([CLOUDFLARED, "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                                   start_new_session=True)  # survives stopping a notebook cell
    t0 = time.time()
    while time.time() - t0 < 60 and not globals().get("TUNNEL_URL"):
        line = TUNNEL_PROC.stdout.readline()
        if not line:
            break
        m = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", line)
        if m:
            TUNNEL_URL = m.group(0)
    threading.Thread(target=lambda: [None for _ in TUNNEL_PROC.stdout], daemon=True).start()  # keep the pipe drained

if globals().get("TUNNEL_URL"):
    print("=" * 60)
    print(f"READY: {TUNNEL_URL}")
    print("Paste this into the website: Image to Video tab → Video server URL.")
    print("This cell is done; the server keeps running in the background.")
    print("Cell 4 shows its log and memory use. To stop it: Runtime → Restart session.")
    print("=" * 60)
else:
    print("Could not start the Cloudflare tunnel. Run this cell again.")

In [ ]:
# Cell 4 (optional): the server's recent log and memory use. Run it again to refresh.
import subprocess
try:
    import psutil
    vm = psutil.virtual_memory()
    print(f"System RAM: {(vm.total - vm.available) / 1e9:.1f} / {vm.total / 1e9:.1f} GB")
except Exception:
    pass
gpu = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
if gpu:
    print(f"GPU memory: {gpu}")
sup = globals().get("SUPERVISOR")
if sup is not None:
    running = sup.proc is not None and sup.proc.poll() is None
    print("Server program:", "running" if running else ("stopped — run cell 3" if sup.gave_up else "starting / restarting"))
    print(f"Crashes in the last 10 minutes: {len([t for t in sup.crashes if __import__('time').time() - t < 600])}")
tp = globals().get("TUNNEL_PROC")
if globals().get("TUNNEL_URL"):
    alive = tp is not None and tp.poll() is None
    print(f"Website URL: {TUNNEL_URL}", "(working)" if alive else "— STOPPED: run cell 3 again and paste its new URL into the website")
print("-" * 60)
try:
    print("".join(open("/content/video_server.log", errors="replace").readlines()[-40:]))
except OSError:
    print("No log yet — run cell 3 first.")